# ACT Finetune: Improving Model Robustness After Verification
#
- This notebook demonstrates a finetuning workflow for neural networks using counterexamples discovered during post-verification.
- After identifying true positive (genuine) adversarial examples via formal verification, we use these samples to further train (finetune) the model.
- This workflow leverages both fuzzing and verification to improve model robustness.

## Setup

In [13]:
import sys, os
act_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, act_root) if act_root not in sys.path else None

# Benchmark weights path
BENCHMARK_ROOT = '/home/guanqinzhang/guanqin/ACT/literature/code/robust-verify-benchmark'
WEIGHTS_DIR = os.path.join(BENCHMARK_ROOT, 'weights', 'experiment_1')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import pandas as pd

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Model Definitions

The benchmark uses a 2-layer MLP: 784 -> 100 -> 100 -> 10

In [2]:
class Flatten(nn.Module):
    """Flatten module for converting [B, C, H, W] to [B, C*H*W]"""
    def forward(self, x):
        return x.view(x.size(0), -1)


def create_mlp_2_100():
    """Create the MLP_2_100 architecture used in the benchmark.
    
    Architecture: 784 -> 100 -> ReLU -> 100 -> ReLU -> 10
    """
    return nn.Sequential(
        Flatten(),
        nn.Linear(784, 100),
        nn.ReLU(),
        nn.Linear(100, 100),
        nn.ReLU(),
        nn.Linear(100, 10)
    )


def load_benchmark_model(model_name):
    """Load pretrained model from benchmark.
    
    Args:
        model_name: One of 'ADV_MLP_B_0.03', 'ADV_MLP_B_0.05', 'ADV_MLP_B_0.1',
                   'NOR_MLP_B', 'LPD_MLP_B_0.1', 'LPD_MLP_B_0.2', 'LPD_MLP_B_0.3', 'LPD_MLP_B_0.4'
    """
    model = create_mlp_2_100()
    weights_path = os.path.join(WEIGHTS_DIR, f'{model_name}.pth')
    
    state_dict = torch.load(weights_path, map_location='cpu')
    model.load_state_dict(state_dict)
    
    return model.to(device).eval()


# Test loading
model = load_benchmark_model('ADV_MLP_B_0.03')
print(f"Model loaded: {model}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

Model loaded: Sequential(
  (0): Flatten()
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=10, bias=True)
)
Total parameters: 89610


## 2. Load MNIST Dataset

In [3]:
# Load MNIST test set
transform = transforms.ToTensor()
testset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(testset, batch_size=100, shuffle=False)

print(f"Test set size: {len(testset)}")
print(f"Number of batches: {len(test_loader)}")

Test set size: 10000
Number of batches: 100


## 3. Evaluation Functions

### 3.1 Clean Accuracy (Test Error)

In [4]:
def evaluate_clean(model, loader):
    """Evaluate clean accuracy.
    
    Returns:
        test_error: Percentage of misclassified samples
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    
    test_error = 1.0 - correct / total
    return test_error * 100  # Return as percentage


# Test
model = load_benchmark_model('ADV_MLP_B_0.03')
test_error = evaluate_clean(model, test_loader)
print(f"ADV_MLP_B_0.03 Test Error: {test_error:.2f}% (expected: 1.53%)")

ADV_MLP_B_0.03 Test Error: 1.53% (expected: 1.53%)


### 3.2 PGD Attack (Lower Bound on Adversarial Error)

Using ACT's front-end PGD mutation.

In [5]:
# Use ACT's front-end PGD attack (reuse instead of custom implementation)
from act.pipeline.finetune import evaluate_pgd


# Test - uses ACT front-end PGDMutation internally
model = load_benchmark_model('ADV_MLP_B_0.03')
pgd_error = evaluate_pgd(model, test_loader, epsilon=0.03, num_steps=40, step_size=0.01)
print(f"ADV_MLP_B_0.03 PGD Error (eps=0.03): {pgd_error:.2f}% (expected: 4.17%)")

[ACT] Auto-detecting project root: ../../../../../../home/guanqinzhang/guanqin/newACT/dual/ACT
[WARN] Gurobi license not found: ../../../../../../home/guanqinzhang/guanqin/newACT/dual/ACT/modules/gurobi/gurobi.lic
[INFO] Please place gurobi.lic in: ../../../../../../home/guanqinzhang/guanqin/newACT/dual/ACT/modules/gurobi
ADV_MLP_B_0.03 PGD Error (eps=0.03): 4.12% (expected: 4.17%)


### 3.3 LP-Greedy / Dual Bounds (Upper Bound on Adversarial Error)

Using ACT's DualTF for certified bound computation.

The LP-Greedy method computes a lower bound on the "margin" for each class:
- `margin[i,j] = output[i] - output[j]` where `i` is the true class
- If `min_j margin[i,j] > 0` for all `j != i`, the sample is **certified robust**
- Otherwise, we cannot certify robustness (may or may not be vulnerable)

In [6]:
from act.pipeline.finetune import ProvableLoss


def evaluate_lp_greedy_act(model, loader, epsilon):
    """Evaluate LP-Greedy (certified) error using ACT's ProvableLoss.
    
    Uses dual bounds to compute certified error rate.
    
    Returns:
        lp_error: Percentage of samples that are NOT certifiably robust
    """
    # Create ProvableLoss for certification
    loss_fn = ProvableLoss(input_clamp=(0.0, 1.0))
    
    total = 0
    not_certified = 0
    
    # Need to convert model to work with ProvableLoss
    # ProvableLoss expects flattened input, but our model has Flatten layer
    # We need to extract the sequential part after Flatten
    
    # Create a wrapper that handles flattening
    class FlatModel(nn.Module):
        def __init__(self, model):
            super().__init__()
            # Extract layers after Flatten
            self.layers = nn.Sequential(*list(model.children())[1:])
        
        def forward(self, x):
            return self.layers(x)
    
    flat_model = FlatModel(model).to(device)
    
    for X, y in tqdm(loader, desc="LP-Greedy"):
        X, y = X.to(device), y.to(device)
        batch_size = X.size(0)
        
        # Flatten X for ProvableLoss
        X_flat = X.view(batch_size, -1)  # [B, 784]
        
        # Compute certified accuracy
        _, metrics = loss_fn(flat_model, X_flat, y, epsilon)
        
        # Count not certified samples
        certified_acc = metrics['certified_acc']
        not_certified += int((1.0 - certified_acc) * batch_size + 0.5)
        total += batch_size
    
    lp_error = not_certified / total
    return lp_error * 100  # Return as percentage


# Test
model = load_benchmark_model('ADV_MLP_B_0.03')
lp_error = evaluate_lp_greedy_act(model, test_loader, epsilon=0.03)
print(f"ADV_MLP_B_0.03 LP-Greedy Error (eps=0.03): {lp_error:.2f}% (expected: 13.40%)")

LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 106.31it/s]

ADV_MLP_B_0.03 LP-Greedy Error (eps=0.03): 18.04% (expected: 13.40%)


## 4. Full Benchmark Evaluation

In [7]:
def evaluate_model(model_name, epsilon):
    """Evaluate a single model configuration.
    
    Returns:
        dict with test_error, pgd_error, lp_error
    """
    print(f"\n{'='*60}")
    print(f"Model: {model_name} | Epsilon: {epsilon}")
    print(f"{'='*60}")
    
    # Load model
    model = load_benchmark_model(model_name)
    
    # Evaluate
    test_error = evaluate_clean(model, test_loader)
    print(f"Test Error: {test_error:.2f}%")
    
    pgd_error = evaluate_pgd(model, test_loader, epsilon)
    print(f"PGD Error: {pgd_error:.2f}%")
    
    lp_error = evaluate_lp_greedy_act(model, test_loader, epsilon)
    print(f"LP-Greedy Error: {lp_error:.2f}%")
    
    return {
        'model': model_name,
        'epsilon': epsilon,
        'test_error': test_error,
        'pgd_error': pgd_error,
        'lp_error': lp_error
    }

In [8]:
# Define benchmark configurations
BENCHMARK_CONFIGS = [
    # ADV trained models
    ('ADV_MLP_B_0.03', 0.03),
    ('ADV_MLP_B_0.05', 0.05),
    ('ADV_MLP_B_0.1', 0.1),
    
    # Normal trained model
    ('NOR_MLP_B', 0.02),
    ('NOR_MLP_B', 0.03),
    ('NOR_MLP_B', 0.05),
    
    # LPD trained models
    ('LPD_MLP_B_0.1', 0.1),
    ('LPD_MLP_B_0.2', 0.2),
    ('LPD_MLP_B_0.3', 0.3),
    ('LPD_MLP_B_0.4', 0.4),
]


In [9]:
# Run full benchmark
results = []

for model_name, epsilon in BENCHMARK_CONFIGS:
    result = evaluate_model(model_name, epsilon)
    results.append(result)


Model: ADV_MLP_B_0.03 | Epsilon: 0.03
Test Error: 1.53%
PGD Error: 4.07%


LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 117.62it/s]


LP-Greedy Error: 18.04%

Model: ADV_MLP_B_0.05 | Epsilon: 0.05
Test Error: 1.62%
PGD Error: 5.91%


LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 117.13it/s]


LP-Greedy Error: 21.87%

Model: ADV_MLP_B_0.1 | Epsilon: 0.1
Test Error: 3.33%
PGD Error: 15.20%


...

LP-Greedy Error: 42.35%

Model: LPD_MLP_B_0.3 | Epsilon: 0.3
Test Error: 39.22%
PGD Error: 56.79%


LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 116.88it/s]


LP-Greedy Error: 69.52%

Model: LPD_MLP_B_0.4 | Epsilon: 0.4
Test Error: 67.97%
PGD Error: 81.83%


LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 117.21it/s]

LP-Greedy Error: 91.61%


## 5. Results Summary

In [10]:
# Create results DataFrame
df = pd.DataFrame(results)

# No EXPECTED_RESULTS, so just show actual results
display_df = df[['model', 'epsilon', 'test_error', 'pgd_error', 'lp_error']]
display_df.columns = ['Model', 'Epsilon', 'Test Error', 'PGD Error', 'LP Error']

print("\n" + "="*100)
print("BENCHMARK RESULTS")
print("="*100)
print(display_df.to_string(index=False))


BENCHMARK RESULTS
         Model  Epsilon  Test Error  PGD Error  LP Error
ADV_MLP_B_0.03     0.03        1.53       4.07     18.04
ADV_MLP_B_0.05     0.05        1.62       5.91     21.87
 ADV_MLP_B_0.1     0.10        3.33      15.20     40.35
     NOR_MLP_B     0.02        2.05       9.78     27.21
     NOR_MLP_B     0.03        2.05      19.35     60.78
     NOR_MLP_B     0.05        2.05      49.48     96.03
 LPD_MLP_B_0.1     0.10        4.09      13.40     19.59
 LPD_MLP_B_0.2     0.20       15.72      33.36     42.35
 LPD_MLP_B_0.3     0.30       39.22      56.79     69.52
 LPD_MLP_B_0.4     0.40       67.97      81.83     91.61


## 7. Single Sample Analysis

Detailed analysis of dual bounds for a single sample.

In [11]:
def analyze_single_sample(model, X, y, epsilon):
    """Detailed analysis of dual bounds for a single sample."""
    # Flatten model
    class FlatModel(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.layers = nn.Sequential(*list(model.children())[1:])
        def forward(self, x):
            return self.layers(x)
    
    flat_model = FlatModel(model).to(device)
    loss_fn = ProvableLoss(input_clamp=(0.0, 1.0))
    
    # Flatten input
    X_flat = X.view(1, -1).to(device)
    y = y.to(device).unsqueeze(0)
    
    # Get bounds
    num_classes = 10
    lb = (X_flat - epsilon).clamp(min=0.0)
    ub = (X_flat + epsilon).clamp(max=1.0)
    
    layer_bounds = loss_fn._forward_bounds(flat_model, lb, ub)
    worst_logits = loss_fn._compute_dual_bounds(flat_model, layer_bounds, lb, ub, y, num_classes)
    
    # Get clean output
    with torch.no_grad():
        clean_output = model(X.to(device))
    
    true_label = y.item()
    
    print(f"\nSingle Sample Analysis (True label: {true_label})")
    print("="*60)
    print(f"Clean output: {clean_output[0].cpu().numpy()}")
    print(f"Clean prediction: {clean_output.argmax(dim=1).item()}")
    print(f"\nDual bounds on margin (output[{true_label}] - output[j]):")
    
    for j in range(num_classes):
        margin_bound = worst_logits[0, j].item()
        actual_margin = (clean_output[0, true_label] - clean_output[0, j]).item()
        status = "CERTIFIED" if margin_bound > 0 else "NOT CERTIFIED"
        if j == true_label:
            status = "(self)"
        print(f"  j={j}: bound={margin_bound:+.4f}, actual={actual_margin:+.4f} {status}")
    
    # Overall certification
    is_certified = loss_fn._compute_certified(worst_logits, y).item()
    print(f"\nOverall: {'CERTIFIED ROBUST' if is_certified else 'NOT CERTIFIED'}")
    
    return worst_logits


# Analyze a sample
model = load_benchmark_model('ADV_MLP_B_0.03')
X_sample, y_sample = next(iter(test_loader))
X_sample = X_sample[0:1]  # First sample
y_sample = y_sample[0]

_ = analyze_single_sample(model, X_sample, y_sample, epsilon=0.03)


Single Sample Analysis (True label: 7)
Clean output: [ -6.7393203   -3.7640643   -1.5716473    0.14634305 -12.906414
  -6.591698   -19.958162    13.325863    -8.630954    -2.7161617 ]
Clean prediction: 7

Dual bounds on margin (output[7] - output[j]):
  j=0: bound=+13.4258, actual=+20.0652 CERTIFIED
  j=1: bound=+11.4495, actual=+17.0899 CERTIFIED
  j=2: bound=+8.5445, actual=+14.8975 CERTIFIED
  j=3: bound=+7.7610, actual=+13.1795 CERTIFIED
  j=4: bound=+19.1703, actual=+26.2323 CERTIFIED
  j=5: bound=+12.7854, actual=+19.9176 CERTIFIED
  j=6: bound=+24.5804, actual=+33.2840 CERTIFIED
  j=7: bound=+0.0000, actual=+0.0000 (self)
  j=8: bound=+15.0421, actual=+21.9568 CERTIFIED
  j=9: bound=+9.5888, actual=+16.0420 CERTIFIED

Overall: CERTIFIED ROBUST


## 8. Training Mode Comparison

Compare how different training methods affect robustness.

In [12]:
# Compare training modes at epsilon=0.1
eps = 0.1

training_modes = [
    ('ADV_MLP_B_0.1', 'ADV (PGD Training)'),
    ('LPD_MLP_B_0.1', 'LPD (Provable Training)'),
]

print(f"\nComparison at epsilon={eps}")
print("="*60)

for model_name, label in training_modes:
    model = load_benchmark_model(model_name)
    
    test_err = evaluate_clean(model, test_loader)
    pgd_err = evaluate_pgd(model, test_loader, eps)
    lp_err = evaluate_lp_greedy_act(model, test_loader, eps)
    
    print(f"\n{label}:")
    print(f"  Test Error: {test_err:.2f}%")
    print(f"  PGD Error (empirical): {pgd_err:.2f}%")
    print(f"  LP-Greedy Error (certified): {lp_err:.2f}%")
    print(f"  Gap (LP - PGD): {lp_err - pgd_err:.2f}%")


Comparison at epsilon=0.1


LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 118.05it/s]



ADV (PGD Training):
  Test Error: 3.33%
  PGD Error (empirical): 15.24%
  LP-Greedy Error (certified): 40.35%
  Gap (LP - PGD): 25.11%


LP-Greedy: 100%|██████████| 100/100 [00:00<00:00, 117.38it/s]


LPD (Provable Training):
  Test Error: 4.09%
  PGD Error (empirical): 13.41%
  LP-Greedy Error (certified): 19.59%
  Gap (LP - PGD): 6.18%
